# Milestone 2 tour: one loop, replaceable search

Milestone 2 keeps task execution fixed while making search replaceable and results fairly comparable. M2.1 introduced replayable archive decisions, linked decision explanations, and a shared policy conformance contract. M2.2 added independent BestOfN. M2.3 added a retained Population with deterministic tournament selection. M2.4 added immutable run summaries, strict comparison, and the canonical policy benchmark.

## 1. Keep the task fixed and swap only search

The seed is evaluated once. `max_trials` counts successor proposals, while the task budget counts all evaluations including that seed.

In [1]:
import meta_evolve as meta
from meta_evolve import domain

TARGET = 12
MOVES = (-2, 1, 5)
BUDGET = meta.Budget(evaluations=9, trials=8)

def propose(parent, context):
    return parent + context.rng.choice(MOVES)

def evaluate(candidate):
    return -abs(candidate - TARGET)

task = meta.Task(
    evaluator=evaluate,
    objectives=(meta.Maximize("score", satisfy=0),),
    budget=BUDGET,
)
policies = {
    "Greedy": meta.Greedy(max_trials=8),
    "BestOfN": meta.BestOfN(max_trials=8),
    "Population": meta.Population(size=3, tournament_size=2, max_trials=8),
}
stores = {name: meta.Storage.in_memory() for name in policies}
runs = {
    name: meta.run(
        meta.Experiment(task=task, seed=0, proposer=propose, search=policy,
                        random_seed=1), storage=stores[name]
    )
    for name, policy in policies.items()
}
print("runs ready:", tuple(runs))

runs ready: ('Greedy', 'BestOfN', 'Population')


In [2]:
for name, result in runs.items():
    summary = result.summary()
    print(
        name,
        f"score={summary.primary_score}",
        f"usage={summary.usage}",
        f"lineage={list(summary.winning_lineage)}",
    )

Greedy score=0 usage=Usage(evaluations=6, trials=5, tokens=0, wall_seconds=0.0) lineage=[0, 5, 10, 11, 12]
BestOfN score=-7 usage=Usage(evaluations=9, trials=8, tokens=0, wall_seconds=0.0) lineage=[0, 5]
Population score=-2 usage=Usage(evaluations=9, trials=8, tokens=0, wall_seconds=0.0) lineage=[0, 1, 2, 7, 8, 9, 14]


In [3]:
result = runs["Greedy"]
summary = result.summary()
print("best:", result.best())
print("primary score:", summary.primary_score)
print("winning lineage:", list(summary.winning_lineage))
print("seed metrics:", dict(result.trials()[0].metrics))

best: 12
primary score: 0
winning lineage: [0, 5, 10, 11, 12]
seed metrics: {'score': -12}


Greedy expands the current best artifact, producing a chain. BestOfN always expands the initial artifact, producing independent spokes of a star. Population retains a rank-ordered top-k archive and expands deterministic tournament winners, producing a tree. These policies change parent selection and retention, not the coordinator or evaluator.

## 2. Compare only compatible runs

`Run.compare()` rejects mismatched evaluator/proposer declarations, objectives, budgets, or starting artifacts. Different policies, policy configurations, and root seeds are allowed. Exact score equality is a tie; usage never breaks one.

In [4]:
for other_name in ("BestOfN", "Population"):
    comparison = runs["Greedy"].compare(runs[other_name])
    winner = comparison.winner.policy.name if comparison.winner else "tie"
    print(
        f"Greedy vs {other_name}: winner={winner}, "
        f"same_random_seed={comparison.same_random_seed}",
    )

Greedy vs BestOfN: winner=greedy, same_random_seed=True
Greedy vs Population: winner=greedy, same_random_seed=True


## 3. Optional kernel inspection: decisions explain themselves

The public callable path ends above. The underscore projection below is an internal diagnostic used here to show the M2 record model: every policy decision has a linked explanation; Population additionally records one `ArchiveUpdate` for every completed evaluation.

In [5]:
from collections import Counter

for name, result in runs.items():
    projection = result._projection()
    kinds = Counter(type(item.decision).__name__ for item in projection.decisions)
    declaration = projection.start.search.policy
    print(
        name,
        f"policy={declaration.name}@{declaration.version}",
        f"decisions={dict(kinds)}",
        f"explanations={len(projection.explanations)}",
    )

Greedy policy=greedy@1 decisions={'TrialSpec': 5, 'Stop': 1} explanations=6
BestOfN policy=best-of-n@1 decisions={'TrialSpec': 8, 'Stop': 1} explanations=9
Population policy=population@1 decisions={'ArchiveUpdate': 9, 'TrialSpec': 8, 'Stop': 1} explanations=18


In [6]:
population = runs["Population"]._projection()
print("Population archive members:", len(population.search_state.archive_members))
print("First three explanations:")
for explanation in population.explanations[:3]:
    print("-", explanation.rationale)

Population archive members: 3
First three explanations:
- admit candidate at population rank 1 of 1
- expand tournament winner at population rank 1 from 1 sampled members
- admit candidate at population rank 1 of 2


## 4. Rebuild the same projection from retained facts

M2 replay means folding the retained in-memory event tuple and reproducing historical policy decisions. It does not yet mean process restart, persistent storage, or resume; those remain later milestones.

In [7]:
from meta_evolve.application import replay_run_projection

result = runs["Population"]
events = stores["Population"].events.read(result.id)
rebuilt = replay_run_projection(result.id, events)
print("retained events:", len(events))
print("projection reproduced:", rebuilt == result._projection())
print("usage reproduced:", rebuilt.search_state.usage)

retained events: 73
projection reproduced: True
usage reproduced: Usage(evaluations=9, trials=8, tokens=0, wall_seconds=0.0)


## Where to go next

Run `main.py` for the concise 40-seed report, then open `policy_comparison.ipynb` for complete distributions and representative star/chain/tree lineages. Those results are a small deterministic illustration in the locked project environment, not broad scientific evidence or a cross-version CPython guarantee. Local callable equality is declaration-based (`module:qualname`) and does not prove behavioral equivalence.